In [0]:
%pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/5.3 MB ? eta -:--:--
   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/5.3 MB 1.9 MB/s eta 0:00:03
   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/5.3 MB 4.1 MB/s eta 0:00:02
   ━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/5.3 MB 9.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━ 2.8/5.3 MB 20.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 5.3/5.3 MB 32.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 30.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:

import pandas as pd
import requests
from io import StringIO
from typing import Tuple, List

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

def _fetch_html(url: str) -> str:
    headers = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0 Safari/537.36"),
        "Accept-Language": "en-US,en;q=0.9",
    }
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    return r.text

def _read_html_tables(html: str, attrs: dict | None = None) -> List[pd.DataFrame]:
    # Use StringIO to pass the HTML string to read_html
    return pd.read_html(StringIO(html), flavor="lxml", attrs=attrs)

def _flatten_columns(df: pd.DataFrame) -> List[str]:
    """
    Returns a list of lowercase string column names, flattening MultiIndex if present.
    """
    cols = []
    for c in df.columns:
        if isinstance(c, tuple):
            # join non-empty levels with a space
            s = " ".join(str(part).strip() for part in c if part is not None and str(part).strip() != "")
        else:
            s = str(c).strip()
        cols.append(s.lower())
    return cols

def get_wikipedia_current_and_changes() -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns (current_constituents_df, changes_df) from Wikipedia.
    Columns in current_constituents_df are standardized to:
      ['symbol', 'name', 'sector', 'subIndustry', 'date_added'] (date_added may be absent).
    The changes_df is returned as-is (you can adapt downstream parsing).
    """
    html = _fetch_html(WIKI_URL)

    # 1) Try to read the constituents table by id
    try:
        current_tables = _read_html_tables(html, attrs={"id": "constituents"})
    except Exception:
        current_tables = []

    current = None
    if current_tables:
        current = current_tables[0]

    # 2) Read all tables for fallback scanning
    all_tables = _read_html_tables(html)

    # If we didn't find current by id, scan for it
    if current is None:
        for t in all_tables:
            cols = _flatten_columns(t)
            if {"symbol", "security", "gics sector"}.issubset(set(cols)):
                current = t
                break

    # Find a plausible "changes" table (has a date column plus added/removed columns)
    changes = None
    for t in all_tables:
        cols = _flatten_columns(t)
        has_date = any("date" in c for c in cols)
        has_added = any(("added" in c) or ("addition" in c) for c in cols)
        has_removed = any(("removed" in c) or ("deletion" in c) for c in cols)
        if has_date and has_added and has_removed:
            changes = t
            break

    # Last-resort fallback if heuristics fail: pick the second table as changes
    if changes is None and len(all_tables) >= 2:
        changes = all_tables[1]

    if current is None or changes is None:
        raise RuntimeError("Could not locate the 'current' or 'changes' tables on the Wikipedia page.")

    # Standardize "current" columns
    current = current.rename(columns={
        "Symbol": "symbol",
        "Security": "name",
        "GICS Sector": "sector",
        "GICS Sub-Industry": "subIndustry",
        "Date added": "date_added",  # may or may not exist in your page version
    })
    # Ensure presence subset (ignore columns not found)
    keep = [c for c in ["symbol", "name", "sector", "subIndustry", "date_added"] if c in current.columns]
    current = current[keep].copy()
    current["symbol"] = current["symbol"].astype(str).str.strip()

    return current, changes
current, changes = get_wikipedia_current_and_changes()

In [0]:
current.to_csv('current_sp500_tickers.csv')

In [0]:
current.shape

(503, 5)

In [0]:
changes.columns = ['_'.join(map(str, col)).strip() for col in changes.columns]

In [0]:
changes.head()

,Effective Date_Effective Date,Added_Ticker,Added_Security,Removed_Ticker,Removed_Security,Reason_Reason
0,"December 22, 2025",CRH,CRH,LKQ,LKQ Corporation,Market capitalization change.[6]
1,"December 22, 2025",CVNA,Carvana,SOLS,Solstice Advanced Materials,Market capitalization change.[6]
2,"December 22, 2025",FIX,Comfort Systems USA,MHK,Mohawk Industries,Market capitalization change.[6]
3,"December 11, 2025",ARES,Ares Management,K,Kellanova,Mars Inc. acquired Kellanova.[7]
4,"November 28, 2025",SNDK,Sandisk,IPG,Interpublic Group,S&P 500 constituent Omnicom Group Inc. acquire...


In [0]:
changes.columns

Index(['Effective Date_Effective Date', 'Added_Ticker', 'Added_Security',
       'Removed_Ticker', 'Removed_Security', 'Reason_Reason'],
      dtype='object')

In [0]:
changes.columns = changes.columns.str.lower()
changes.columns

Index(['effective date_effective date', 'added_ticker', 'added_security',
       'removed_ticker', 'removed_security', 'reason_reason'],
      dtype='object')

In [0]:
changes.rename(columns={'effective date_effective date': 'effective_date'}, inplace=True)
changes.rename(columns={'reason_reason': 'reason'}, inplace=True)
changes.to_csv('changes_sp500_tickers.csv')

In [0]:
def create_historical_sp500_list(current_sp500_df, changes_df):
    """
    Combines current SP500 list and a changes log to create a historical table
    with date_added and date_removed for every stock that was ever in the index,
    using the effective_date for both actions.
    """
    
    # --- Preparation ---
    historical_df = current_sp500_df.copy()
    # Initialize all current companies as having 'NA' removed date
    historical_df['date_removed'] = 'NA' 
    
    # Convert all relevant date columns to datetime objects early
    historical_df['date_added'] = pd.to_datetime(historical_df['date_added'], errors='coerce')
    changes_df['effective_date'] = pd.to_datetime(changes_df['effective_date'], errors='coerce')
    
    # Ensure 'date_removed' starts as a datetime column to handle NaTs correctly later
    historical_df['date_removed'] = pd.to_datetime(historical_df['date_removed'], errors='coerce')

    # --- Step 1 & 2: Process Removals and Add new historical rows ---
    for _, row in changes_df.iterrows():
        removed_ticker = row['removed_ticker']
        # The effective_date is the date of removal
        removal_date = row['effective_date'] 
        
        if pd.isna(removed_ticker):
            continue
            
        # Scenario A: Ticker is in the current list -> Update its removal date
        if removed_ticker in historical_df['symbol'].values:
            mask = historical_df['symbol'] == removed_ticker
            historical_df.loc[mask, 'date_removed'] = removal_date
            
        # Scenario B: Ticker is NOT in current list (historical-only stock) -> Add a new row
        else:
            new_row_data = {
                'symbol': removed_ticker,
                'name': row['removed_security'],
                'sector': 'Unknown', # Sector information is usually missing in the changes table
                'subIndustry': 'Unknown',
                'date_added': pd.NaT, # We don't know the added date from this row yet
                'date_removed': removal_date
            }
            # Use pd.concat to add the new row DataFrame
            historical_df = pd.concat([historical_df, pd.DataFrame([new_row_data])], ignore_index=True)

    # --- Step 3: Backfill date_added using the 'added_ticker' column ---
    for _, row in changes_df.iterrows():
        added_ticker = row['added_ticker']
        # The effective_date is also the date of addition
        addition_date = row['effective_date']

        if pd.isna(added_ticker):
            continue
            
        if added_ticker in historical_df['symbol'].values:
            # Only update date_added if it is currently missing (NaT) or 'Unknown'
            mask = (historical_df['symbol'] == added_ticker) & (historical_df['date_added'].isna())
            historical_df.loc[mask, 'date_added'] = addition_date
            
        # If an added ticker isn't in the list by now, something is wrong with input data, 
        # but for robustness we could add it here too.

    # --- Step 4: Final Cleanup and Formatting (Error-proofed) ---
    
    # Use the .dt accessor to format dates safely to strings, then fill NaT values
    historical_df['date_added'] = historical_df['date_added'].dt.strftime('%Y-%m-%d').fillna('Unknown')
    historical_df['date_removed'] = historical_df['date_removed'].dt.strftime('%Y-%m-%d').fillna('NA')

    return historical_df.sort_values('symbol').reset_index(drop=True)


In [0]:
def get_sp500_list_at_date(history_df, target_date_str):
    """
    Filters the historical S&P 500 DataFrame to list companies that 
    were active constituents on the target date.
    
    Args:
        history_df (pd.DataFrame): The DataFrame with symbol, name, date_added, date_removed.
        target_date_str (str): The date to filter on, in 'YYYY-MM-DD' format.
    
    Returns:
        pd.DataFrame: A DataFrame of active companies on that date.
    """
    
    # 1. Ensure all date columns are proper datetime objects
    target_date = pd.to_datetime(target_date_str)
    
    # Convert 'date_added' to datetime if it isn't already
    history_df['date_added'] = pd.to_datetime(history_df['date_added'])
    
    # Convert 'date_removed' to datetime, handling the 'NA' strings gracefully
    # 'NA' will be converted to a NaT (Not a Time) value.
    history_df['date_removed_dt'] = pd.to_datetime(history_df['date_removed'], errors='coerce')
    
    # 2. Define the filtering conditions
    
    # Condition A: The stock must have been added ON or BEFORE the target date.
    added_before_target = history_df['date_added'] <= target_date
    
    # Condition B: The stock must have been removed AFTER the target date,
    # OR it must have a 'NaT' (meaning it's still current/was current at the end of the input data).
    removed_after_target = (history_df['date_removed_dt'] > target_date) | (history_df['date_removed_dt'].isna())
    
    # 3. Apply both conditions
    active_on_date_df = history_df[added_before_target & removed_after_target].copy()
    
    # Clean up the temporary column before returning
    active_on_date_df = active_on_date_df.drop(columns=['date_removed_dt'])
    
    return active_on_date_df

In [0]:
historical_sp500_table = create_historical_sp500_list(current, changes)
historical_sp500_table.head()

,symbol,name,sector,subIndustry,date_added,date_removed
0,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,2000-06-05,NA
1,AA,Alcoa,Unknown,Unknown,Unknown,2016-11-01
2,AAL,American Airlines Group,Unknown,Unknown,2015-03-23,2024-09-23
3,AAP,Advance Auto Parts,Unknown,Unknown,2015-07-08,2023-08-25
4,AAPL,Apple Inc.,Information Technology,"Technology Hardware, Storage & Peripherals",1982-11-30,NA


In [0]:
historical_sp500_table[historical_sp500_table.date_added == 'Unknown']

,symbol,name,sector,subIndustry,date_added,date_removed
1,AA,Alcoa,Unknown,Unknown,Unknown,2016-11-01
9,ABS,Albertsons,Unknown,Unknown,Unknown,2006-06-02
11,ACAS,American Capital,Unknown,Unknown,Unknown,2009-03-03
12,ACE,Chubb,Unknown,Unknown,Unknown,2016-01-19
16,ADCT,ADC Telecommunications,Unknown,Unknown,Unknown,2007-07-02
...,...,...,...,...,...,...
830,XLNX,Xilinx,Unknown,Unknown,Unknown,2022-02-15
832,XRAY,Dentsply Sirona,Unknown,Unknown,Unknown,2024-04-03
833,XRX,Xerox,Unknown,Unknown,Unknown,2021-03-22
834,XTO,XTO Energy,Unknown,Unknown,Unknown,2010-06-28


In [0]:
changes.removed_ticker

0       LKQ
1      SOLS
2       MHK
3         K
4       IPG
       ... 
383     SUN
384     USL
385     MCK
386     HNG
387     AYE
Name: removed_ticker, Length: 388, dtype: object

In [0]:
universe_ticker = list(set(changes.added_ticker.to_list() + changes.removed_ticker.to_list() + current.symbol.to_list()))
universe_ticker

['FL',
 'LIFE',
 'CVNA',
 'CPRI',
 'LXK',
 'AYI',
 'BLK',
 'APOL',
 'TFX',
 'ABBV',
 'MMM',
 'ADI',
 'JNY',
 'ABMD',
 'NE',
 'ESRX',
 'LUK',
 'PBI',
 'LOW',
 'BRO',
 'AMG',
 'MAT',
 'UDR',
 'FLIR',
 'HSP',
 'MIL',
 'BDX',
 'LEN',
 'MU',
 'SHLD',
 'SLB',
 'TRIP',
 'NBL',
 'VAR',
 'WBA',
 'RSH',
 'AIZ',
 'JBHT',
 'FRT',
 'PH',
 'COF',
 'EP',
 'AMP',
 'GIS',
 'CMA',
 'CMI',
 'SPG',
 'FRX',
 'CVG',
 'SUN',
 'HPQ',
 'TROW',
 'ANR',
 'BAC',
 'PSX',
 'DLPH',
 'PXD',
 'HOT',
 'MMI',
 'BXLT',
 'BEN',
 'RL',
 'ACN',
 'FBHS',
 'WBD',
 'CCK',
 'VICI',
 'WY',
 'RIG',
 'PNW',
 'CHD',
 'NYT',
 'TMUS',
 'MFE',
 'BK',
 'CZR',
 'CMCSK',
 'MOH',
 'BKR',
 'EMN',
 'FHN',
 'AMCR',
 'ROL',
 'YUM',
 'COG',
 'SBL',
 'TECH',
 'QEP',
 'PWR',
 'MOS',
 'NYX',
 'JCI',
 'CRL',
 'SMCI',
 'PTC',
 'PRGO',
 'EVRG',
 'APP',
 'CSC',
 'WFC',
 'AXP',
 'TPR',
 'SNPS',
 'ADSK',
 'JKHY',
 'VLO',
 'TEG',
 'FII',
 'CERN',
 'LLL',
 'MOLX',
 'WYN',
 'WYNN',
 'FMC',
 'DNR',
 'TRGP',
 'SCG',
 'CTSH',
 'STZ',
 'COTY',
 'TMO',
 'NDSN'

In [0]:
values_with_dots = [symbol for symbol in universe_ticker if '.' in str(symbol)]
print(values_with_dots)

['BRK.B', 'BF.B']


In [0]:
values_with_dots = [symbol for symbol in universe_ticker if 'BF' in str(symbol)]
print(values_with_dots)

['BF.B']


In [0]:
# The map of {old_value: new_value}
duplicate_map = {
    'GOOG': 'GOOGL',  # Alphabet Class C -> Class A
    'FOX': 'FOXA',    # Fox Corp Class B -> Class A
    'NWS': 'NWSA',    # News Corp Class B -> Class A
    'BRK.A': 'BRK.B'  # Berkshire Class A -> Class B (Safety check)
}

# List comprehension: "Give me the new value if it exists, else give me the original"
updated_universe_tickers = list(set([duplicate_map.get(ticker, ticker) for ticker in universe_ticker]))

df_universe_tickers = pd.DataFrame(updated_universe_tickers, columns=['ticker'])
df_universe_tickers.to_csv("universe_tickers.csv")

In [0]:
df_universe_tickers.shape

(861, 1)